<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/L5/bayesian_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Inference: Protein Biomarker for Parkinson's Disease

**7.571 — Quantitative Biology | Lecture 05**

---

## The Scenario

We believe we've identified a new protein marker that is predictive of Parkinson's Disease. If this protein is observed in the brain, the patient will develop Parkinson's.

We have brain tissue samples from **20 patients** who developed the disease. Each sample is scored as **positive (1)** or **negative (0)** for the protein.

### Questions:
1. What fraction of Parkinson's patients have this protein?
2. How confident are we in this estimate?
3. How does our estimate change as we get more data?
4. Does our starting assumption (prior) matter?

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

---
## Part 1 — The Data

In [ ]:
# Binary staining results: 1 = protein present, 0 = absent
protein_staining_result = np.array([1,0,1,0,1,1,0,0,1,0,0,1,0,1,1,1,1,1,1,0])

n_patients = len(protein_staining_result)
n_positive = protein_staining_result.sum()
n_negative = n_patients - n_positive
point_estimate = n_positive / n_patients

print(f"Patients tested:     {n_patients}")
print(f"Positive staining:   {n_positive}")
print(f"Negative staining:   {n_negative}")
print(f"Point estimate (p̂):  {point_estimate:.2f}")

---
## Part 2 — Why the Beta Distribution?

We need a **prior distribution** for θ (the true probability of staining positive). We use the **Beta distribution** because:

1. **It lives on [0, 1]** — perfect for a probability
2. **It's flexible** — two parameters (a, b) can represent many different beliefs
3. **Conjugate prior** — when combined with Binomial data, the posterior is also Beta (makes the math clean)

The parameters **a** and **b** act like "pseudo-counts": Beta(a=3, b=10) is like saying *"before seeing data, it's as if I'd already seen 3 positives and 10 negatives."*

In [ ]:
# Different priors encode different beliefs
xs = np.linspace(0, 1, 1000)
fig, ax = plt.subplots()

for a, b, label in [(1,1,'Uniform (a=1, b=1)'), (10,10,'Centered (a=10, b=10)'), (2,8,'Skeptical (a=2, b=8)')]:
    ax.plot(xs, stats.beta(a, b).pdf(xs), lw=2, label=label)

ax.set_xlabel('θ (probability of staining)')
ax.set_ylabel('Density')
ax.set_xlim(0, 1)
ax.legend()
ax.set_title('Prior Distributions')
plt.show()

---
## §3 — The Bayesian Update

We want:

$$P(\theta \mid \text{data}) \propto P(\text{data} \mid \theta) \times P(\theta)$$

With a Beta prior and Binomial data, the update is simple:

- **a_new = a + (number of positives)**
- **b_new = b + (number of negatives)**

The posterior is Beta(a_new, b_new). Your prior pseudo-counts get combined with real observed counts.

**MAP (Maximum A Posteriori)** = the peak of the posterior — our best single estimate of θ.

### 3a. Uniform Prior

Start agnostic: Beta(1, 1) — no prior knowledge.

In [ ]:
a_prior, b_prior = 1, 1
prior = stats.beta(a_prior, b_prior)

# Update: just add the counts
a_post = a_prior + n_positive
b_post = b_prior + n_negative
posterior = stats.beta(a_post, b_post)

map_estimate = xs[posterior.pdf(xs).argmax()]
ci = posterior.interval(0.95)

print(f"Prior:      Beta({a_prior}, {b_prior})")
print(f"Posterior:  Beta({a_post}, {b_post})")
print(f"MAP:        {map_estimate:.3f}")
print(f"95% range:  ({ci[0]:.3f}, {ci[1]:.3f})")

# Plot
fig, ax = plt.subplots()
ax.plot(xs, prior.pdf(xs), lw=1.5, ls='--', color='gray', label=f'Prior: Beta({a_prior},{b_prior})')
ax.plot(xs, posterior.pdf(xs), lw=2.5, color='C0', label=f'Posterior: Beta({a_post},{b_post})')
ax.axvline(map_estimate, color='black', lw=1, ls=':', label=f'MAP = {map_estimate:.2f}')
ax.set_xlabel('θ'); ax.set_ylabel('Density'); ax.set_xlim(0,1)
ax.legend(); ax.set_title('Uniform Prior → Posterior')
plt.show()

With a uniform prior, the MAP equals the simple fraction (12/20 = 0.60). The data completely determines the result.

### 3b. Skeptical Prior

Suppose we know staining for this protein is uncommon in the general population. Prior: Beta(3, 10) — centered around ~23%.

In [ ]:
a_skep, b_skep = 3, 10
prior_skep = stats.beta(a_skep, b_skep)

a_post_skep = a_skep + n_positive
b_post_skep = b_skep + n_negative
posterior_skep = stats.beta(a_post_skep, b_post_skep)

map_skep = xs[posterior_skep.pdf(xs).argmax()]

print(f"Prior:      Beta({a_skep}, {b_skep}) — centered ~{a_skep/(a_skep+b_skep):.0%}")
print(f"Posterior:  Beta({a_post_skep}, {b_post_skep})")
print(f"MAP:        {map_skep:.3f}  (vs {map_estimate:.3f} with uniform prior)")
print(f"\nThe skeptical prior pulled the estimate down.")

# Plot
fig, ax = plt.subplots()
ax.plot(xs, prior_skep.pdf(xs), lw=1.5, ls='--', color='gray', label=f'Prior: Beta({a_skep},{b_skep})')
ax.plot(xs, posterior_skep.pdf(xs), lw=2.5, color='C0', label=f'Posterior: Beta({a_post_skep},{b_post_skep})')
ax.axvline(map_skep, color='black', lw=1, ls=':', label=f'MAP = {map_skep:.2f}')
ax.set_xlabel('θ'); ax.set_ylabel('Density'); ax.set_xlim(0,1)
ax.legend(); ax.set_title('Skeptical Prior → Posterior')
plt.show()

---
## Part 4 — New Data Arrives

Six months later, we collect **19 more samples**. We use the old posterior as the new prior — this is how science builds on itself.

In [ ]:
# New data
new_data = np.array([0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1])
new_positive = new_data.sum()
new_negative = len(new_data) - new_positive

# Chain: old posterior → new prior → new posterior
a_new = a_post_skep + new_positive
b_new = b_post_skep + new_negative

original_prior = stats.beta(a_skep, b_skep)
first_posterior = stats.beta(a_post_skep, b_post_skep)
final_posterior = stats.beta(a_new, b_new)
map_final = xs[final_posterior.pdf(xs).argmax()]

print(f"Original prior:     Beta({a_skep}, {b_skep})")
print(f"After 20 patients:  Beta({a_post_skep}, {b_post_skep})")
print(f"After 39 patients:  Beta({a_new}, {b_new})")
print(f"Final MAP:          {map_final:.3f}")

# Plot
fig, ax = plt.subplots()
ax.plot(xs, original_prior.pdf(xs), lw=1.5, ls=':', color='gray', label=f'Original prior')
ax.plot(xs, first_posterior.pdf(xs), lw=1.5, ls='--', color='C1', label=f'After 20 patients')
ax.plot(xs, final_posterior.pdf(xs), lw=2.5, color='C0', label=f'After 39 patients')
ax.axvline(map_final, color='black', lw=1, ls=':', label=f'MAP = {map_final:.2f}')
ax.set_xlabel('θ'); ax.set_ylabel('Density'); ax.set_xlim(0,1)
ax.legend(); ax.set_title('Sequential Updating')
plt.show()

print(f"\nNotice: the posterior gets taller and narrower. More data = more certainty.")

---
## §5 — Prior Convergence

*"But priors are subjective!"* — Let's see what happens with enough data.

In [ ]:
# Three very different starting beliefs, same data
prior_configs = [
    (2, 8, 'Skeptical'),
    (1, 1, 'Uniform'),
    (8, 2, 'Optimistic'),
]

np.random.seed(42)
true_rate = 0.60
sample_sizes = [5, 20, 100, 500]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for idx, n in enumerate(sample_sizes):
    ax = axes[idx]
    data = np.random.binomial(1, true_rate, n)
    npos, nneg = data.sum(), n - data.sum()

    for a, b, label in prior_configs:
        post = stats.beta(a + npos, b + nneg)
        ax.plot(xs, post.pdf(xs), lw=2, label=label if idx == 0 else None)

    ax.set_title(f'n = {n}')
    ax.set_xlabel('θ')
    ax.set_xlim(0, 1)
    if idx == 0: ax.set_ylabel('Density')

axes[0].legend(fontsize=9)
plt.suptitle('With enough data, different priors converge', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

| Concept | What we learned |
|---------|----------------|
| **Bayes' Theorem** | P(H\|D) ∝ P(D\|H) × P(H) — update beliefs with data |
| **Prior** | Encodes what you knew before; Beta distribution for probabilities |
| **Posterior** | Updated belief after seeing data; taller = more certain |
| **MAP** | Peak of the posterior — best single estimate |
| **Conjugate prior** | Beta × Binomial → Beta (just add counts!) |
| **Sequential updating** | Old posterior becomes new prior |
| **Prior convergence** | With enough data, the prior doesn't matter much |